# Step 05 — LLM classification

Step 04 decided *which* buildings are places of activity. This notebook asks,
for each of them, **what activities happen inside and what kind of building it
is**, using an LLM on the evidence step 04 collected.

| | |
|---|---|
| **Reads** | `data/output/04_buildings_enriched.gpkg` — layer `buildings` (39,786 rows, 43 columns) and `building_pois` |
| **Writes** | nothing yet — decided section by section, with approval |
| **Needs** | `pyogrio`, `pandas`; the LLM client comes with section 3 |

## State of this notebook

Built **one step at a time**, each run and inspected before the next is written.

| step | | status |
|---|---|---|
| **05.1** | **The columns** — which the LLM reads, which only the model needs, which are noise for this step | **implemented** |
| **05.2** | **The prompt input** — one record per building from the LLM columns, plus neighbour context where a building has none of its own | pending |
| **05.3** | **The prompt and the output schema** — activity labels, Bosserhof class, confidence, reason | pending |
| **05.4** | **Routing** — per building where there is evidence, per signature where there is only class, land and size | pending |
| **05.5** | **The calls, and the validation against the rule baseline** | pending |

## Why an LLM, and what it is for

The previous pipeline settled this on an annotated set: the LLM reached 78.5 %
against 57.7 % for the rule table, and the whole difference was business-name
world knowledge — the model knows what *Deutsche Bank*, *Ernsting's family* or
*Tischlerei Holzteam* are. This step keeps that design and feeds the model what
step 04 has assembled per building: the POIs on it with names and uses, the
site around it, the cadastre class and name, the OSM footprint tag and name,
the land under it, and its size.

In [1]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError('Cannot find the pipeline root (the folder containing config.py). '
                       f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.')
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import time
import numpy as np
import pandas as pd
import pyogrio

from config import (
    ENRICHED_BUILDINGS_FILE,
    LLM_COLUMN_ROLES, LLM_INPUT_COLS, LLM_MODEL_COLS, LLM_DROPPED_COLS,
)
from lib.checks import require_file, require_non_empty, require_unique, require_cols

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 70)

print('Root :', ROOT_DIR)
print('Input:', ENRICHED_BUILDINGS_FILE.name)

Root : C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
Input: 04_buildings_enriched.gpkg


## 1. The columns

The enriched layer carries 43 columns. Not all of them belong in a prompt, and
some must not be there. `LLM_COLUMN_ROLES` in `config.py` assigns every column
one of three roles, with the reason next to it; this section reads the layer,
checks that **every column is assigned** — a column added or removed in step 04
stops this step until it is classified — and shows what each one holds.

**The LLM sees** what describes what happens inside, in three blocks of
falling trust:

1. *what is inside* — `poi_uses`, `poi_names`, `site_uses`, `site_names`
2. *what the building is* — `label_en`, `name`, `osm_tag` (one field from
   `osm_twin_tag` on ALKIS rows and `osm_building` on OSM rows), `osm_twin_name`
3. *where and how big* — `alkis_landuse`, `alkis_landuse_detail` (the parcel's
   coded kind: education and science, health, power plant, campsite ...),
   `osm_landuse`, `city`, `area_m2`, `height_top_max_m`

**Only the model needs** the keys, the weight and the baseline: `building_id`,
`alkis_id`, `ags`, `function`, `volume_3d_m3`, `source`, `n_pois`, `n_sites`,
`address`, `activities`. Two of these are deliberately withheld from the
prompt. `activities` is the rule table's answer from the ALKIS class; shown, it
anchors the model to the rule, hidden, it is the baseline to validate against.
`address` proves a mailbox, not an activity, and the model has no lookup at
call time — asked about an address it would invent a tenant.

**Noise for this step** is step-03 provenance and QA, geometry bookkeeping,
columns already folded into others, the filter's own provenance (`rescued`,
`rescued_by`), and QGIS legend text. They stay in the step 04 output; they do
not enter step 05.

The three name columns stay separate on purpose. Measured on the layer: 1,345
buildings have an OSM footprint name that no POI carries, 1,521 an ALKIS name
and nothing else, and 10,466 have POI names but no footprint name. A name
repeated by three sources is a name three sources agree on; merged into one
list the model would not know who said what.

In [2]:
require_file(ENRICHED_BUILDINGS_FILE, 'enriched buildings (step 04)')
_layers = [l[0] for l in pyogrio.list_layers(ENRICHED_BUILDINGS_FILE)]
if 'buildings' not in _layers:
    raise AssertionError(f'layer "buildings" missing from {ENRICHED_BUILDINGS_FILE.name}: {_layers}')

print('Reading the enriched buildings (attributes only) ...', flush=True)
t0 = time.perf_counter()
bld = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='buildings', read_geometry=False)
print(f'  ok  {len(bld):,} buildings x {len(bld.columns)} columns  [{time.perf_counter() - t0:,.1f}s]')
require_non_empty(bld, 'enriched buildings')
require_unique(bld, 'building_id', 'enriched buildings')

# --- every column must have a role, and every role a column -------------------
_layer_cols = set(bld.columns)
_role_cols = set(LLM_COLUMN_ROLES)
_unassigned = sorted(_layer_cols - _role_cols)
_missing = sorted(_role_cols - _layer_cols)
if _unassigned or _missing:
    raise AssertionError(
        'config.LLM_COLUMN_ROLES and the layer disagree - '
        f'columns in the layer without a role: {_unassigned}; '
        f'roles for columns the layer no longer has: {_missing}. '
        'Classify them in config before running this step.')
print(f'  ok  all {len(_layer_cols)} columns have a role: '
      f'{len(LLM_INPUT_COLS)} the LLM sees, {len(LLM_MODEL_COLS)} model only, {len(LLM_DROPPED_COLS)} dropped here')

# --- what each column holds ------------------------------------------------------
def _example(s):
    v = s.dropna()
    if v.empty:
        return ''
    v = v.iloc[min(7, len(v) - 1)]
    return str(v)[:60]

review = pd.DataFrame({
    'role':      [LLM_COLUMN_ROLES[c][0] for c in bld.columns],
    'filled_%':  (bld.notna().mean() * 100).round(1).to_numpy(),
    'distinct':  bld.nunique().to_numpy(),
    'example':   [_example(bld[c]) for c in bld.columns],
    'why':       [LLM_COLUMN_ROLES[c][1] for c in bld.columns],
}, index=pd.Index(bld.columns, name='column'))
_order = {'llm': 0, 'model': 1, 'drop': 2}
review = review.iloc[np.argsort([_order[r] for r in review['role']], kind='stable')]
for _role, _title in (('llm', 'THE LLM SEES'), ('model', 'ONLY THE MODEL NEEDS'), ('drop', 'NOISE FOR THIS STEP - not read')):
    print()
    print(f'=== {_title} ({int((review["role"] == _role).sum())} columns)')
    print(review.loc[review['role'] == _role, ['filled_%', 'distinct', 'example', 'why']].to_string())

# --- how much of the layer has how much to say --------------------------------------
_has_inside = bld['poi_uses'].notna() | bld['site_uses'].notna()
_has_name = bld['name'].notna() | bld['osm_twin_name'].notna()
_tag = bld['osm_twin_tag'].fillna(bld['osm_building'])
_has_tag = _tag.notna() & (_tag != 'yes')
print()
print('  ..  what the LLM will have to work with, per building:')
print(f'        POIs or a site on it           {int(_has_inside.sum()):>7,}  ({100 * _has_inside.mean():.1f} %)')
print(f'        a name but no POI or site      {int((~_has_inside & _has_name).sum()):>7,}')
print(f'        only an informative OSM tag    {int((~_has_inside & ~_has_name & _has_tag).sum()):>7,}')
print(f'        class, land use and size only  {int((~_has_inside & ~_has_name & ~_has_tag).sum()):>7,}  '
      f'({100 * (~_has_inside & ~_has_name & ~_has_tag).mean():.1f} %) - the per-signature group, section 4')

  ok  04_buildings_enriched.gpkg (42.0 MB)
Reading the enriched buildings (attributes only) ...


  ok  39,786 buildings x 43 columns  [0.4s]
  ok  enriched buildings: 39,786 rows
  ok  enriched buildings.building_id: unique and non-null (39,786)
  ok  all 43 columns have a role: 15 the LLM sees, 10 model only, 18 dropped here

=== THE LLM SEES (15 columns)
                      filled_%  distinct                                             example                                                                                                                               why
column                                                                                                                                                                                                                        
name                      12.0      2528                                           Gärtnerei                                         the cadastre's own label (Tischlerei, Grundschule, Vereinsheim); OSM name on the gap rows
city                     100.0       134                             

## Where this leaves us

Section 1 is a decision, not a transformation: nothing is written. The layer has
43 columns; 15 go to the LLM (14 prompt fields, since the two OSM tag columns
merge), 10 ride along for the model, 18 stop here. The check above turns that
decision into a contract: if step 04 changes its columns, this notebook refuses
to run until config says what the new column is for.

### Next

**05.2, the prompt input.** One record per building from the 15 columns, with
`osm_tag` merged, numbers rounded, and the three name fields kept apart. No
information is added from outside the building: a neighbour field (the nearest
named building within 30 m) was measured, proposed and rejected on 2026-09-14
— what the model reads about a building must be what the sources say about
*that* building. The parcel's coded kind (`alkis_landuse_detail`) *is* about
that building's own ground and was added the same day. The 6,052 buildings
with only class, land use and size are classified per signature (section 4),
not enriched.
